# HJB Optimal Depth Calibration to LOBSTER Data
## AAPL 2012-06-21 · Level-5 Limit Order Book · 10-second Bars

This notebook calibrates the **closed-form HJB optimal half-spread** $\delta^*_t$ to
intraday LOBSTER order-book data and measures the error against the observed market spread:

$$E_t \;:=\; \delta^*_t \;-\; \delta_{\text{market},t}$$

### Closed-form strategy (eq. `final_strategy` in the paper)

$$
\delta^{\pm*}_t = \delta^\pm_0
  + B(\delta^\pm_0;\,\kappa^\pm)\Bigl[
      \underbrace{\pm\,\mathbb{S}^\pm_\lambda\,\mathbb{E}\!\left(\int_t^T\alpha_u\,du\right)}_{\text{Trend correction}}
      +\underbrace{\phi\!\left(\pm\,\mathbb{S}^\pm_\lambda b_\phi + (1\mp 2q)\,|c_\phi(t,V)|\right)}_{\text{Inventory \& volatility risk}}
    \Bigr] + o(\varsigma)
$$

We evaluate the **symmetric zero-inventory zero-trend baseline** ($q=0$, $\alpha_t=0$,
$\lambda^+=\lambda^-$), which collapses to:

$$\boxed{\delta^*_t = \delta_0 + B(\delta_0;\kappa)\cdot\phi\cdot|c_\phi(t,V_t)|}$$

where **every coefficient is re-evaluated at each 10-second bar** using the empirical $V_t$.

---

### Term-by-term definitions

| Symbol | Formula | Units | Notes |
|--------|---------|-------|-------|
| $\kappa_\text{emp}$ | fitted from execution depths | $\$/^{-1}$ | decay of $\lambda(\delta)=Ae^{-\kappa\delta}$ |
| $\delta_0$ | $1/(2\kappa_\text{emp})$ | \$ | AS indifference half-spread |
| $B_\text{exact}$ | $e^{+0.5}/\kappa$ | \$ | from FOC: $1/(\kappa\,h(\delta_0))$; $= e\cdot B_\text{AS}$ |
| $c_\phi(t,V_t)$ | see below | dimensionless | expected integrated variance $t\to T$ |

$$c_\phi(t,V) = -\!\left[\theta_V(T-t) + (V_t-\theta_V)\,\frac{1-e^{-\kappa_V(T-t)}}{\kappa_V}\right]$$

**Key design choices documented here:**
- $\theta_V = 0.04$ is the **model** CIR long-run mean, not the empirical daily mean.  
  Using the empirical mean makes $(V_t-\theta_V)\approx 0$ everywhere, killing the vol-dependent variation.
- Annualisation uses $\sqrt{\text{ticks/sec}\times 23400\times 252}$, **not** $\sqrt{252\times 23400}$  
  (the latter treats time in seconds rather than ticks, giving a 3.59× underestimate of $\sigma_t$).

---

### Structural gap

The HJB formula captures only **inventory risk**.  The market spread also prices:
- Adverse selection (Kyle / Glosten–Milgrom)
- Tick-size discreteness (AAPL quotes in \$0.01 increments)
- Queue-position risk and latency premia

So $\delta^*_t < \delta_{\text{market},t}$ on average — this is economically interpretable, not a bug.  
The **variation** of $\delta^*_t$ correctly tracks empirical volatility with correlation $\approx 0.82$.


## 0 · Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import zipfile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import norm, pearsonr

sns.set_theme(style="whitegrid")
PAL = sns.color_palette("deep")
%matplotlib inline

# ── Data path ─────────────────────────────────────────────────────────────────
# Update this to wherever the LOBSTER zip file lives.
ZIP_PATH = 'LOBSTER_SampleFile_AAPL_2012-06-21_5.zip'

# ── Model hyperparameters (from MarketParams / paper) ─────────────────────────
KAPPA_V   = 4.0    # CIR mean-reversion speed  κ_V
THETA_V   = 0.04   # CIR long-run variance  θ_V  — MODEL value, NOT empirical mean
T_HORIZON = 1.0    # normalised trading horizon (full day = 1)
PHI       = 5.0    # inventory-risk aversion φ  (Agent 2 in the simulation)

# ── Data-processing parameters ────────────────────────────────────────────────
N_LEVELS       = 5       # number of LOB price levels in the file
VOL_WINDOW_SEC = 60.0    # look-back window for rolling vol proxy (seconds)
DAILY_SECONDS  = 23400.0 # seconds in a 6.5-hour NYSE trading day
BAR_SEC        = 10.0    # bar width for downsampling (seconds)
FILL_BINS      = 40      # histogram bins for fill-probability fit

print("Configuration loaded.")
print(f"  KAPPA_V={KAPPA_V}, THETA_V={THETA_V}, PHI={PHI}")
print(f"  BAR_SEC={BAR_SEC}s, VOL_WINDOW={VOL_WINDOW_SEC}s")


## 1 · Load LOBSTER Data

The zip contains two headerless CSVs:

| File | Description |
|------|-------------|
| `orderbook_5.csv` | LOB state after every event. Columns (repeated for each of the 5 levels): `ask_price_i`, `ask_size_i`, `bid_price_i`, `bid_size_i`. Prices in units of $\tfrac{1}{10000}$ \$. |
| `message_5.csv` | Event log. Columns: `time` (s since midnight), `type`, `order_id`, `size`, `price` ($\tfrac{1}{10000}$ \$), `direction` (+1=buy, −1=sell). |

Message types used here: **4** = visible limit-order execution, **5** = hidden execution.  
Rows where prices hit the LOBSTER sentinel value (±999 999 999 in raw units) are removed.


In [ ]:
def load_lobster(zip_path: str):
    """Load, clean and return the orderbook and message DataFrames."""
    ob_fname  = 'AAPL_2012-06-21_34200000_57600000_orderbook_5.csv'
    msg_fname = 'AAPL_2012-06-21_34200000_57600000_message_5.csv'

    with zipfile.ZipFile(zip_path) as z:
        with z.open(ob_fname)  as f: ob  = pd.read_csv(f, header=None)
        with z.open(msg_fname) as f: msg = pd.read_csv(f, header=None)

    # ── Column names ──────────────────────────────────────────────────────────
    ob_cols = []
    for i in range(1, N_LEVELS + 1):
        ob_cols += [f'ask_price_{i}', f'ask_size_{i}',
                    f'bid_price_{i}', f'bid_size_{i}']
    ob.columns  = ob_cols
    msg.columns = ['time', 'type', 'order_id', 'size', 'price', 'direction']

    # ── Convert 1/10000 $ → $ ─────────────────────────────────────────────────
    for i in range(1, N_LEVELS + 1):
        ob[f'ask_price_{i}'] /= 10000.0
        ob[f'bid_price_{i}'] /= 10000.0
    msg['price'] /= 10000.0

    ob['time'] = msg['time'].values   # attach timestamp from message file

    # ── Drop LOBSTER dummy / halt rows (sentinel price > 999 999) ─────────────
    valid = (ob['ask_price_1'] < 999999) & (ob['bid_price_1'] > -999999)
    ob    = ob[valid].copy().reset_index(drop=True)
    msg   = msg[valid].copy().reset_index(drop=True)

    # ── Derived columns ───────────────────────────────────────────────────────
    ob['mid']             = (ob['ask_price_1'] + ob['bid_price_1']) / 2.0
    ob['half_spread_ask'] = ob['ask_price_1'] - ob['mid']
    ob['half_spread_bid'] = ob['mid'] - ob['bid_price_1']

    return ob, msg


ob, msg = load_lobster(ZIP_PATH)

print(f"Valid snapshots : {len(ob):,}")
print(f"Time range      : {ob['time'].min():.0f} – {ob['time'].max():.0f} s after midnight")
print(f"Mid-price range : ${ob['mid'].min():.4f} – ${ob['mid'].max():.4f}")
print(f"Ask half-spread : mean=${ob['half_spread_ask'].mean():.5f}  "
      f"std=${ob['half_spread_ask'].std():.5f}")
ob.head(3)


## 2 · Rolling Volatility Proxy $\sqrt{V_t}$

For each tick $i$ we compute the sample standard deviation of log-returns
$\log(S_{k}/S_{k-1})$ over all ticks in the preceding 60-second window:

$$\hat{\sigma}_i = \text{std}\!\left(\{\log(S_k/S_{k-1})\}_{t_k\in[t_i-60,\,t_i]}\right)$$

This is a **per-tick** standard deviation.  To convert to annualised units
(matching the Heston $\theta_V=0.04$ scale):

$$\sqrt{V_t} \approx \hat{\sigma}_i \times \underbrace{\sqrt{\text{ticks/sec}\times 23400\times 252}}_{\text{ANNUAL\_FACTOR}}$$

> **Common bug:** using $\sqrt{252\times23400}$ treats time in *seconds* rather than *ticks*,
> underestimating the annual factor by $\sqrt{\text{ticks/sec}}\approx3.59\times$,
> which makes $V_\text{emp}\approx13\times$ too small and kills all vol-driven variation in $\delta^*_t$.


In [ ]:
def compute_rolling_vol(t_arr: np.ndarray, mid: np.ndarray,
                        window_sec: float = VOL_WINDOW_SEC):
    """
    Returns
    -------
    sqrt_V : annualised volatility σ_t = √V_t  (shape N)
    V      : annualised variance   V_t = σ_t²  (shape N)
    """
    log_ret = np.concatenate([[0.0], np.log(mid[1:] / mid[:-1])])

    # Correct tick-based annualisation
    ticks_per_sec = len(t_arr) / (t_arr[-1] - t_arr[0])
    annual_factor = np.sqrt(ticks_per_sec * DAILY_SECONDS * 252.0)

    # Rolling window: for each tick find all ticks in [t_i - window_sec, t_i]
    vol_proxy = np.empty(len(t_arr))
    for i in range(len(t_arr)):
        lo = np.searchsorted(t_arr, t_arr[i] - window_sec, side='left')
        w  = log_ret[lo : i + 1]
        vol_proxy[i] = w.std() if len(w) > 2 else np.nan

    vol_proxy = pd.Series(vol_proxy).ffill().bfill().values   # fill warm-up NaNs

    sqrt_V = vol_proxy * annual_factor
    V      = sqrt_V ** 2
    return sqrt_V, V, annual_factor, ticks_per_sec


sqrt_V_tick, V_tick, annual_factor, ticks_per_sec = compute_rolling_vol(
    ob['time'].values, ob['mid'].values
)

print(f"ticks/sec     = {ticks_per_sec:.2f}")
print(f"ANNUAL_FACTOR = {annual_factor:.0f}  (tick-based, correct)")
print(f"√V_t  : mean={sqrt_V_tick.mean():.4f}  std={sqrt_V_tick.std():.4f}  "
      f"range=[{sqrt_V_tick.min():.4f}, {sqrt_V_tick.max():.4f}]")
print(f"V_t   : mean={V_tick.mean():.4f}  (model θ_V = {THETA_V})")


## 3 · Fill-Probability Fit $\lambda(\delta) = A\,e^{-\kappa\delta}$

We recover the empirical fill-probability decay $\kappa$ by:

1. Identifying all executions in the message file (types 4 & 5).
2. Computing the **execution depth** from the contemporaneous mid-price:
   - Buyer-initiated (+1): $\delta = p_\text{exec} - S_t$ (ask side, above mid)
   - Seller-initiated (−1): $\delta = S_t - p_\text{exec}$ (bid side, below mid)
3. Fitting $\lambda(\delta) = A\,e^{-\kappa\delta}$ to the histogram of depths via nonlinear least squares.

### B coefficient derivation

The optimality first-order condition (FOC) gives $B = 1/(\kappa\,h(\delta_0))$ where
$h(\delta)=e^{-\kappa\delta}$.  Substituting $\delta_0 = 1/(2\kappa)$:

$$B_\text{exact} = \frac{e^{+0.5}}{\kappa} = e\cdot B_\text{AS}, \qquad
B_\text{AS} = \frac{e^{-0.5}}{\kappa}$$

The Avellaneda–Stoikov convention uses $B_\text{AS}$; the paper's FOC gives $B_\text{exact}$.
We use $B_\text{exact}$ throughout.


In [ ]:
def fit_fill_probability(ob: pd.DataFrame, msg: pd.DataFrame):
    """
    Returns kappa_emp, A_fit, delta0, B_exact, (bin_centers, bin_counts).
    """
    mid = ob['mid'].values

    exec_mask  = msg['type'].isin([4, 5])
    exec_pos   = np.where(exec_mask)[0]
    exec_price = msg.loc[exec_mask, 'price'].values
    exec_dir   = msg.loc[exec_mask, 'direction'].values
    exec_mid   = mid[exec_pos]

    depth = np.where(exec_dir == 1,
                     exec_price - exec_mid,   # ask side
                     exec_mid  - exec_price)  # bid side
    depth = depth[depth > 0]

    bins          = np.linspace(0, np.percentile(depth, 95), FILL_BINS)
    counts, edges = np.histogram(depth, bins=bins)
    bin_centers   = (edges[:-1] + edges[1:]) / 2.0
    nz            = counts > 0

    popt, _ = curve_fit(
        lambda d, A, k: A * np.exp(-k * d),
        bin_centers[nz], counts[nz],
        p0=[counts[nz].max(), 25.0], maxfev=5000
    )
    A_fit, kappa_emp = popt

    delta0  = 1.0 / (2.0 * kappa_emp)
    B_as    = np.exp(-kappa_emp * delta0) / kappa_emp   # AS convention
    B_exact = 1.0 / (kappa_emp * np.exp(-kappa_emp * delta0))  # FOC

    assert abs(B_exact / B_as - np.e) < 1e-6, "B ratio sanity check failed"

    print(f"n executions  = {len(depth):,}")
    print(f"κ_emp         = {kappa_emp:.4f}  (1/$)")
    print(f"δ_0 = 1/(2κ)  = ${delta0:.5f}")
    print(f"B_AS          = ${B_as:.6f}")
    print(f"B_exact       = ${B_exact:.6f}   (= e × B_AS ✓, ratio = {B_exact/B_as:.6f})")
    return kappa_emp, A_fit, delta0, B_exact, (bin_centers[nz], counts[nz])


kappa_emp, A_fit, delta0, B_exact, (bin_c, bin_cnt) = fit_fill_probability(ob, msg)

# Quick plot of the fit
fig, ax = plt.subplots(figsize=(7, 4))
bw = bin_c[1] - bin_c[0]
ax.bar(bin_c, bin_cnt, width=bw*0.9, color=PAL[2], alpha=0.55, label='Empirical counts')
d_fine = np.linspace(0, bin_c.max(), 300)
ax.plot(d_fine, A_fit*np.exp(-kappa_emp*d_fine), color=PAL[3], lw=2.5,
        label=fr'Fit: $\kappa$={kappa_emp:.2f}, $\delta_0$=${delta0:.4f}')
ax.set_xlabel(r'Execution depth $\delta$ ($)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(r'Fill Probability  $\lambda(\delta)=Ae^{-\kappa\delta}$', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()


## 4 · Downsample to 10-Second Bars

To make the analysis tractable and reduce tick-level noise, we downsample to
fixed-width time bars.  For each bar $[t,\,t+\Delta t)$ we take the **last
LOB snapshot** in the bar, capturing the state of the book at bar close.

This gives:
- $S_t$ : mid-price at bar close
- $\sqrt{V_t}$, $V_t$ : empirical volatility proxy at bar close  
- $\delta_{\text{market},t}$ : observed best ask half-spread $= P_a^{(1)} - S_t$


In [ ]:
def make_bars(ob: pd.DataFrame, sqrt_V: np.ndarray, V: np.ndarray,
              bar_sec: float = BAR_SEC) -> pd.DataFrame:
    """Downsample tick data to fixed-width bars (last-observation-carried-forward)."""
    t_arr = ob['time'].values
    mid   = ob['mid'].values
    t0, t1 = t_arr[0], t_arr[-1]

    bar_edges = np.arange(t0, t1, bar_sec)

    # Last tick index whose timestamp falls before bar_edge + bar_sec
    bar_idx = np.clip(
        np.searchsorted(t_arr, bar_edges + bar_sec, side='right') - 1,
        0, len(t_arr) - 1
    )

    bars = pd.DataFrame({
        'time'        : bar_edges,
        't_norm'      : (bar_edges - t0) / (t1 - t0),      # ∈ [0, 1)
        'tau'         : 1.0 - (bar_edges - t0) / (t1 - t0),# time remaining T−t
        'mid'         : mid[bar_idx],
        'ask_price_1' : ob['ask_price_1'].values[bar_idx],
        'bid_price_1' : ob['bid_price_1'].values[bar_idx],
        'sqrt_V'      : sqrt_V[bar_idx],
        'V'           : V[bar_idx],
    })

    bars['delta_market_ask'] = bars['ask_price_1'] - bars['mid']
    bars['delta_market_bid'] = bars['mid'] - bars['bid_price_1']
    bars['delta_market_sym'] = (bars['delta_market_ask'] + bars['delta_market_bid']) / 2.0

    print(f"{len(bars)} bars  (bar width = {bar_sec:.0f} s)")
    print(f"δ_market ask : mean=${bars['delta_market_ask'].mean():.5f}  "
          f"std=${bars['delta_market_ask'].std():.5f}")
    return bars


bars = make_bars(ob, sqrt_V_tick, V_tick)
bars[['time','t_norm','tau','mid','sqrt_V','V',
      'delta_market_ask','delta_market_bid']].head(5)


## 5 · HJB Optimal Half-Spreads $\delta^*_t$

We evaluate the closed-form spread at each bar:

$$\delta^*_t = \delta_0 + B \cdot \phi \cdot |c_\phi(t,\,V_t)|$$

### $c_\phi(t, V_t)$: expected integrated variance

From the Feynman–Kac representation under CIR dynamics
(using $\mathbb{E}_t[V_u]=\theta_V+(V_t-\theta_V)e^{-\kappa_V(u-t)}$):

$$c_\phi(t,V) = -\!\left[
  \underbrace{\theta_V(T-t)}_{\text{det. term}}
  +\underbrace{(V_t-\theta_V)\,\frac{1-e^{-\kappa_V(T-t)}}{\kappa_V}}_{\text{vol-deviation term}}
\right]$$

**Both components are re-evaluated at every bar** using the current empirical $V_t$:

- **Deterministic term** $\theta_V\tau$: decays linearly from $\theta_V T$ at open to 0 at close.
- **Vol-deviation term**: amplifies $|c_\phi|$ when $V_t>\theta_V$ (elevated vol), with a
  mean-reversion discount $(1-e^{-\kappa_V\tau})/\kappa_V$ that shrinks as $\tau\to 0$.

> **Why $\theta_V=0.04$ and not the empirical mean?**  
> $\theta_V$ is a structural CIR parameter, not a day-specific calibration target.  
> Using the daily mean $\bar{V}_\text{emp}$ makes $(V_t-\theta_V)\approx 0$ everywhere,
> eliminating the volatility-adaptive widening at the open.


In [ ]:
def compute_optimal_spreads(bars: pd.DataFrame, delta0: float, B: float) -> pd.DataFrame:
    """
    Add δ*_t and error columns to the bar DataFrame.
    All terms re-evaluated at each bar using the empirical V_t.
    """
    tau = bars['tau'].values
    V_t = bars['V'].values

    # c_φ(t,V_t) — split into two components for decomposition plots
    det_component = THETA_V * tau                                                # ≥ 0
    dev_component = (V_t - THETA_V) * (1.0 - np.exp(-KAPPA_V * tau)) / KAPPA_V  # ± signed

    c_phi     = -(det_component + dev_component)   # always ≤ 0
    abs_c_phi = np.abs(c_phi)

    # Optimal half-spread: δ*_t = δ_0 + B·φ·|c_φ(t,V_t)|
    delta_star = delta0 + B * PHI * abs_c_phi

    bars = bars.copy()
    bars['c_phi']          = c_phi
    bars['abs_c_phi']      = abs_c_phi
    bars['c_phi_det']      = det_component          # θ_V·τ
    bars['c_phi_vol_dev']  = np.abs(dev_component)  # |(V_t−θ_V)·(1−e^{−κτ})/κ|
    bars['delta_star']     = delta_star

    bars['E_ask'] = bars['delta_star'] - bars['delta_market_ask']
    bars['E_bid'] = bars['delta_star'] - bars['delta_market_bid']
    bars['E_sym'] = (bars['E_ask'] + bars['E_bid']) / 2.0

    # Diagnostics
    corr_sv, _ = pearsonr(delta_star, bars['sqrt_V'].values)
    corr_mv, _ = pearsonr(bars['delta_market_ask'].values, bars['sqrt_V'].values)
    rmse       = np.sqrt((bars['E_sym']**2).mean())

    print(f"δ_0                 = ${delta0:.5f}")
    print(f"B (e^+0.5/κ)        = ${B:.6f}")
    print(f"φ                   = {PHI}")
    print(f"B·φ                 = {B*PHI:.5f}  (multiplier on |c_φ|)")
    print()
    print(f"|c_φ| range         : [{abs_c_phi.min():.5f}, {abs_c_phi.max():.5f}]  "
          f"mean={abs_c_phi.mean():.5f}")
    print(f"δ* range            : [${delta_star.min():.5f}, ${delta_star.max():.5f}]  "
          f"std=${delta_star.std():.5f}")
    print(f"δ_market mean       = ${bars['delta_market_ask'].mean():.5f}")
    print()
    print(f"Corr(δ*,  √V_t)     = {corr_sv:.4f}   ← vol-tracking quality of the model")
    print(f"Corr(δ_mkt,√V_t)    = {corr_mv:.4f}")
    print()
    print(f"Mean E_t            = ${bars['E_sym'].mean():.5f}  "
          f"← structural gap (model lacks adverse selection)")
    print(f"Std  E_t            = ${bars['E_sym'].std():.5f}")
    print(f"RMSE E_t            = ${rmse:.5f}")
    return bars


bars = compute_optimal_spreads(bars, delta0, B_exact)
bars[['t_norm','tau','V','sqrt_V','c_phi','abs_c_phi',
      'delta_star','delta_market_ask','E_sym']].head(5)


## 6 · Diagnostic Figures

### 6.1 · Market Context: Mid-Price and Volatility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
t_plot = bars['t_norm'].values

# Mid-price
axes[0].plot(t_plot, bars['mid'], color=PAL[0], lw=0.8, alpha=0.85)
axes[0].set_title('Mid-Price  $S_t$', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Normalised time  $t/T$', fontsize=11)
axes[0].set_ylabel('Price ($)', fontsize=11)
axes[0].set_xlim(0, 1)

# √V_t
axes[1].plot(t_plot, bars['sqrt_V'], color=PAL[1], lw=0.9, alpha=0.80,
             label=r'Empirical $\sqrt{V_t}$')
axes[1].axhline(np.sqrt(THETA_V), color='firebrick', ls='--', lw=2.0,
                label=fr'Model $\sqrt{{	heta_V}}$ = {np.sqrt(THETA_V):.3f}')
axes[1].axhline(bars['sqrt_V'].mean(), color=PAL[1], ls=':', lw=1.5,
                label=f'Empirical mean = {bars["sqrt_V"].mean():.3f}')
axes[1].set_title(r'Volatility Proxy  $\sqrt{V_t}$  (60-s rolling, annualised)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Normalised time  $t/T$', fontsize=11)
axes[1].set_ylabel(r'$\sqrt{V_t}$', fontsize=11)
axes[1].set_ylim(0, min(bars['sqrt_V'].quantile(0.98) * 1.3, 2.5))
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()


### 6.2 · $|c_\phi(t,V_t)|$ Decomposition

The two additive components of $|c_\phi|$ — the deterministic decay $\theta_V\tau$
and the volatility-deviation term $|(V_t-\theta_V)(1-e^{-\kappa_V\tau})/\kappa_V|$ —
are plotted separately to show how elevated opening volatility ($V_t\gg\theta_V$) drives
the spike in $|c_\phi|$, and hence in $\delta^*_t$, at the start of the session.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_plot, bars['abs_c_phi'],   color=PAL[0], lw=2.2, label=r'$|c_\phi(t,V_t)|$ total')
ax.plot(t_plot, bars['c_phi_det'],   color=PAL[3], lw=1.5, ls='--',
        label=r'Det: $	heta_V\,	au$')
ax.plot(t_plot, bars['c_phi_vol_dev'], color=PAL[1], lw=1.5, ls=':',
        label=r'Vol dev: $|(V_t-	heta_V)rac{1-e^{-\kappa_V	au}}{\kappa_V}|$')
ax.set_title(r'$|c_\phi(t,V_t)|$ Decomposition  (re-evaluated each bar)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Normalised time  $t/T$', fontsize=11)
ax.set_ylabel(r'$|c_\phi|$  (ann. var × time)', fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()


### 6.3 · $\delta^*_t$ vs $\delta_{\text{market},t}$

Left: overlay of the HJB optimal and market half-spreads — the shaded region is the
structural gap attributable to adverse selection and tick-size effects absent from the model.  
Right: $\delta^*_t$ zoomed with $\sqrt{V_t}$ on the right axis, confirming that the
vol-driven widening is correctly captured.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Overlay
axes[0].plot(t_plot, bars['delta_market_ask'], color=PAL[1], lw=1.2, alpha=0.75,
             label=r'$\delta_{m market}$  (ask half-spread)')
axes[0].plot(t_plot, bars['delta_star'], color=PAL[0], lw=2.0,
             label=r'$\delta^*_t$  HJB optimal')
axes[0].fill_between(t_plot, bars['delta_star'], bars['delta_market_ask'],
                     alpha=0.15, color=PAL[4],
                     label='Gap: adverse sel. + tick size')
axes[0].set_title(r'$\delta^*_t$ vs $\delta_{{m market},t}$', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Normalised time  $t/T$', fontsize=11)
axes[0].set_ylabel('Half-spread ($)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)

# Zoomed with twin vol axis
ax2 = axes[1].twinx()
ax2.fill_between(t_plot, bars['sqrt_V'], alpha=0.12, color=PAL[1])
ax2.plot(t_plot, bars['sqrt_V'], color=PAL[1], lw=0.8, alpha=0.5)
ax2.set_ylabel(r'$\sqrt{V_t}$  (right)', fontsize=11, color=PAL[1])
ax2.tick_params(axis='y', labelcolor=PAL[1])
ax2.set_ylim(0, min(bars['sqrt_V'].quantile(0.98) * 2.5, 3.0))
axes[1].plot(t_plot, bars['delta_star'], color=PAL[0], lw=2.2, zorder=5,
             label=r'$\delta^*_t$  (left)')
axes[1].axhline(delta0, color='grey', ls=':', lw=1.5,
                label=fr'$\delta_0$ = ${delta0:.4f}')
axes[1].set_title(r'$\delta^*_t$ Zoomed  —  vol-driven variation', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Normalised time  $t/T$', fontsize=11)
axes[1].set_ylabel(r'$\delta^*_t$  ($)', fontsize=11)
axes[1].legend(fontsize=9, loc='upper right')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()


### 6.4 · Error Analysis  $E_t = \delta^*_t - \delta_{\text{market},t}$

The error is almost uniformly negative: the HJB spread sits below the market spread
because the model has no adverse-selection term.  The rolling mean (right panel) shows
that the gap is largest at the open (high vol, market widens more than model) and
narrows during the quiet midday period.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Time-series
axes[0].plot(t_plot, bars['E_sym'], color=PAL[4], lw=0.7, alpha=0.55,
             label=r'$E_t$  (sym avg)')
roll = pd.Series(bars['E_sym'].values).rolling(30, center=True).mean().values
axes[0].plot(t_plot, roll, color=PAL[3], lw=2.5, label='30-bar rolling mean')
axes[0].axhline(0, color='black', ls='--', lw=1.2)
axes[0].axhline(bars['E_sym'].mean(), color=PAL[3], ls=':', lw=1.5,
                label=f'Day mean = ${bars["E_sym"].mean():.4f}')
axes[0].fill_between(t_plot, bars['E_sym'], 0, where=bars['E_sym']>0,
                     interpolate=True, color=PAL[0], alpha=0.20)
axes[0].fill_between(t_plot, bars['E_sym'], 0, where=bars['E_sym']<0,
                     interpolate=True, color=PAL[1], alpha=0.20)
axes[0].set_title(r'Error  $E_t = \delta^*_t - \delta_{{m market},t}$',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Normalised time  $t/T$', fontsize=11)
axes[0].set_ylabel(r'$E_t$  ($)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)

# Distribution
E = bars['E_sym'].dropna().values
axes[1].hist(E, bins=60, color=PAL[0], alpha=0.60, density=True, label='$E_t$')
mu_E, sig_E = E.mean(), E.std()
x_g = np.linspace(E.min(), E.max(), 300)
axes[1].plot(x_g, norm.pdf(x_g, mu_E, sig_E), color=PAL[3], lw=2.5,
             label=fr'$\mathcal{{N}}$({mu_E:.4f}, {sig_E:.4f}²)')
axes[1].axvline(0, color='black', ls=':', lw=1.5)
axes[1].set_title(r'Error Distribution  $E_t$', fontsize=12, fontweight='bold')
axes[1].set_xlabel(r'$E_t$  ($)', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


### 6.5 · Error vs Volatility

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
v_cap = bars['sqrt_V'].quantile(0.97)
mask  = bars['sqrt_V'] <= v_cap
sc = ax.scatter(bars.loc[mask, 'sqrt_V'], bars.loc[mask, 'E_sym'],
                c=t_plot[mask.values], cmap='plasma', alpha=0.35, s=5, rasterized=True)
plt.colorbar(sc, ax=ax, label='Normalised time  $t/T$')
zfit = np.polyfit(bars.loc[mask, 'sqrt_V'].values, bars.loc[mask, 'E_sym'].values, 1)
sv_r = np.linspace(0, v_cap, 100)
ax.plot(sv_r, np.polyval(zfit, sv_r), 'r-', lw=2,
        label=f'OLS slope = {zfit[0]:.3f}')
ax.axhline(0, color='black', ls='--', lw=1.0)
ax.set_title(r'Error vs Volatility  $E_t$ vs $\sqrt{V_t}$', fontsize=12, fontweight='bold')
ax.set_xlabel(r'$\sqrt{V_t}$  (annualised)', fontsize=11)
ax.set_ylabel(r'$E_t$  ($)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

corr_ev, _ = pearsonr(bars.loc[mask, 'sqrt_V'].values, bars.loc[mask, 'E_sym'].values)
print(f"Corr(E_t, √V_t) = {corr_ev:.4f}")
print("Negative slope: high-vol periods widen the market spread more than the model,")
print("consistent with adverse-selection costs rising with volatility.")


## 7 · Summary Statistics

In [ ]:
E = bars['E_sym'].values
rmse = np.sqrt((E**2).mean())

summary = pd.DataFrame({
    'Value': [
        # Fitted
        f"{kappa_emp:.4f}",
        f"${delta0:.5f}",
        f"${B_exact:.6f}",
        f"{B_exact*PHI:.5f}",
        # Vol proxy
        f"{THETA_V:.4f}",
        f"{bars['V'].mean():.4f}",
        f"{bars['sqrt_V'].mean():.4f}",
        f"{bars['sqrt_V'].std():.4f}",
        # c_phi
        f"{bars['abs_c_phi'].mean():.5f}",
        f"{bars['abs_c_phi'].min():.5f}",
        f"{bars['abs_c_phi'].max():.5f}",
        # delta*
        f"${bars['delta_star'].min():.5f}",
        f"${bars['delta_star'].max():.5f}",
        f"${bars['delta_star'].mean():.5f}",
        f"${bars['delta_star'].std():.5f}",
        f"{np.corrcoef(bars['delta_star'], bars['sqrt_V'])[0,1]:.4f}",
        # delta_market
        f"${bars['delta_market_ask'].mean():.5f}",
        f"${bars['delta_market_ask'].std():.5f}",
        f"{np.corrcoef(bars['delta_market_ask'], bars['sqrt_V'])[0,1]:.4f}",
        # error
        f"${E.mean():.5f}",
        f"${E.std():.5f}",
        f"${rmse:.5f}",
        f"${E.min():.5f}",
        f"${E.max():.5f}",
        f"{(E>0).mean():.4f}",
    ]
}, index=[
    'κ_emp  [1/$]',
    'δ_0 = 1/(2κ)  [$]',
    'B_exact = e^{+0.5}/κ  [$]',
    'B·φ  (spread correction multiplier)',
    'θ_V  (model CIR long-run)',
    'Mean empirical V_t',
    'Mean empirical √V_t',
    'Std  empirical √V_t',
    'Mean |c_φ|',
    'Min  |c_φ|  (end of day)',
    'Max  |c_φ|  (open, high vol)',
    'Min  δ*_t  [$]',
    'Max  δ*_t  [$]',
    'Mean δ*_t  [$]',
    'Std  δ*_t  [$]',
    'Corr(δ*, √V_t)',
    'Mean δ_market  [$]',
    'Std  δ_market  [$]',
    'Corr(δ_market, √V_t)',
    'Mean E_t  [$]  ← structural gap',
    'Std  E_t  [$]',
    'RMSE E_t  [$]',
    'Min  E_t  [$]',
    'Max  E_t  [$]',
    'Fraction E_t > 0',
])

summary


## 8 · Save Bar-Level Results to CSV

In [ ]:
cols_out = ['time', 't_norm', 'tau', 'mid', 'sqrt_V', 'V',
            'c_phi', 'abs_c_phi', 'c_phi_det', 'c_phi_vol_dev',
            'delta_star', 'delta_market_ask', 'delta_market_bid',
            'E_ask', 'E_bid', 'E_sym']

out_csv = 'lobster_hjb_bars.csv'
bars[cols_out].to_csv(out_csv, index=False, float_format='%.7f')
print(f"Saved {len(bars)} bars to {out_csv}")
bars[cols_out].describe().round(5)
